## setup + imports

In [98]:
# === imports ===
import os, numpy as np, pandas as pd, torch, random
from tqdm.auto import tqdm
from utils.data_utils.load_data import load_dataset, extract_target_properties
from utils.evaluate_utils.load_paths import load_paths
from utils.evaluate_utils.load_models import load_config, load_decoder, load_fNN_model
from utils.evaluate_utils.sampling import *
from utils.evaluate_utils.structure_constraints import enforce_theta_domain, filter_S_candidates
from utils.evaluate_utils.error import compute_tensor_error
from utils.test_utils import *

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## load best parallel models and fNN

In [99]:
# === load best parallel models ===
MODELS = [
    ("001",18), 
    ("010",6), 
    ("100",9), 
    ("011",16), 
    ("101",5), 
    ("110",3), 
    ("111",5)
]

decoders   = []
configs    = []
P_means    = []
P_stds     = []
S_means    = []
S_stds     = []

for tag, trial in MODELS:
    paths = load_paths(
        TRIAL=trial,
        THETA_MODEL=True,
        THETA_PATTERN=tag,
        verbose=False
    )

    # load config + decoder
    config = load_config(paths["config_path"])
    decoder = load_decoder(config, paths["decoder_path"], flow_type=config.get("FLOW_TYPE","planar"),
                       trial=trial, device=device)

    # load per-model stats (fixed across trials for a given tag)
    Pm = np.load(paths["P_mean_path"]);  Ps = np.load(paths["P_std_path"])
    Sm = np.load(paths["S_mean_path"]);  Ss = np.load(paths["S_std_path"])
    Ps = np.where(Ps < 1e-8, 1.0, Ps)
    Ss = np.where(Ss < 1e-8, 1.0, Ss)

    # stash
    decoders.append(decoder)
    configs.append(config)
    P_means.append(Pm); P_stds.append(Ps)
    S_means.append(Sm); S_stds.append(Ss)

# === load max's forward model ===
fNN = load_fNN_model()

/Users/ellielin/Desktop/dresden/inverse_design_spinodoids/spinodoid_cvae/utils/evaluate_utils/load_models.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  decoder.load_s

✅ Loaded decoder from trial 18
✅ Loaded decoder from trial 6
✅ Loaded decoder from trial 9
✅ Loaded decoder from trial 16
✅ Loaded decoder from trial 5
✅ Loaded decoder from trial 3
✅ Loaded decoder from trial 5
✅ Loaded Max's forward model


## load test dataset + choose target sample

In [100]:
# === load test data ===
TEST_CSV = "data/test/dataset_test_x1000.csv"
P_all, S_all, C_all = load_dataset(TEST_CSV)
P_all = P_all.to(device)
S_all = S_all.to(device)
print(f"✅ Test set loaded: N={len(P_all)} | P: {tuple(P_all.shape)} | S: {tuple(S_all.shape)} | C: {C_all.shape}")

# === config for evaluation ===
PROB_THRESHOLD = 0.10
SAMPLES_PER_P  = 1000
BW_MODE        = "auto"

# === pick test item ===
target_idx = 0
S_true = S_all[target_idx].detach().cpu().numpy().flatten()
P_true = P_all[target_idx].detach().cpu().numpy().flatten()
C_true = C_all[target_idx]

print(f"✅ Target index {target_idx}")
print(f"   - S_true: {np.array2string(S_true, formatter={'float_kind': lambda x: f'{x:.4f}'})}")
print(f"   - P_true: {np.array2string(P_true, formatter={'float_kind': lambda x: f'{x:.4f}'})}")

✅ Test set loaded: N=1000 | P: (1000, 9) | S: (1000, 4) | C: (1000, 3, 3, 3, 3)
✅ Target index 0
   - S_true: [34.4427 87.3751 0.0000 0.5033]
   - P_true: [0.2574 0.0823 0.0838 0.2489 0.0847 0.2660 0.0839 0.0862 0.0863]


## run target sample on all models

1. fNN baseline: fNN(S_true) = P_hat_fNN
2. for each theta model: 
        - theta(P_true) = S_hat_theta
        - fNN(S_hat_theta) = P_hat_theta
3. compare P_hat_fNN and P_hat_theta to P_true

In [101]:
# === baseline: fNN evaluated at S_true (table) ===
S_true_tf = np.expand_dims(S_true, axis=(0, 1))
C_fNN_true = fNN(S_true_tf).numpy().reshape(1,3,3,3,3)[0]

err_base = compute_tensor_error(C_true, C_fNN_true)   # C-tensor error (fraction)
base_status = "✅ PASS" if err_base < 0.08 else "❌ FAIL"

df_fnn = pd.DataFrame([{
    "Ŝ":         format_array(S_true),                 # using S_true as Ŝ_fNN reference
    "ΔS":         format_array(np.zeros_like(S_true)),  # S_true - S_true
    "error":      f"{err_base:.4%}",                    # percent format
    "status":     base_status,
}])

print(f"\n⚪ fNN baseline prediction: fNN(S_true) = P_hat_fNN")
display(df_fnn)

# === evaluate target sample on each parallel model ===
for midx, (tag, trial) in enumerate(MODELS):
    decoder    = decoders[midx]
    cfg        = configs[midx]
    P_mean     = P_means[midx]; P_std  = P_stds[midx]
    S_mean     = S_means[midx]; S_std  = S_stds[midx]
    latent_dim = int(cfg["LATENT_DIM"])

    # normalize P for this model
    P_norm = (P_true - P_mean) / (P_std + 1e-8)
    P_norm_t = torch.tensor(P_norm, dtype=torch.float32, device=device).unsqueeze(0)

    # sample Ŝ (normalized)
    S_hats_norm = get_S_hats(decoder, P_norm_t, latent_dim, num_samples=SAMPLES_PER_P, seed=SEED, device=device)

    # peaks + bandwidth
    if isinstance(BW_MODE, str) and BW_MODE.lower() == "auto":
        S_hat_peaks_norm, bw_used = extract_peaks_with_bandwidth(
            S_hats_norm, use_auto_bandwidth=True, target_range=(1, 10), verbose=False
        )
    else:
        bw_used = float(BW_MODE)
        S_hat_peaks_norm = get_S_hat_peaks(S_hats_norm, bandwidth=bw_used)

    # sort + probability filter
    S_hat_peaks_norm, probs, _ = sort_and_select_peaks_by_probability(
        S_hats_norm, S_hat_peaks_norm, bw_used, prob_threshold=PROB_THRESHOLD, verbose=False
    )

    # denorm + constraints
    S_hat_peaks_unnorm = S_hat_peaks_norm * S_std + S_mean
    S_hat_peaks_unnorm = enforce_theta_domain(S_hat_peaks_unnorm)
    S_hat_peaks_unnorm = filter_S_candidates(S_hat_peaks_unnorm)

    # forward each candidate → Ĉ and compute error
    rows = []
    for S_hat in S_hat_peaks_unnorm:
        dS = S_hat - S_true
        S_hat_tf = np.expand_dims(S_hat, axis=(0, 1))
        C_pred = fNN(S_hat_tf).numpy().reshape(1,3,3,3,3)[0]
        err  = float(compute_tensor_error(C_true, C_pred))
        stat = "✅" if err < 0.08 else "❌ FAIL"
        interesting = "✅" if (np.abs(S_hat[:3] - S_true[:3]) >= 5.0).any() else "❌"
        rows.append({
            "Ŝ":    format_array(S_hat),
            "ΔS":   format_array(dS),
            "error":    f"{err:.4%}",
            "status":   stat,
            "interesting": interesting
        })

    df = pd.DataFrame(rows)
    print(f"\n⚪ {tag} — {len(df)} candidates | bw_used={bw_used:.3f}")
    display(df)


⚪ fNN baseline prediction: fNN(S_true) = P_hat_fNN


,Ŝ,ΔS,error,status
0,"[34.44271, 87.37508, 0.00000, 0.50335]","[0.00000, 0.00000, 0.00000, 0.00000]",1.6910%,✅ PASS



⚪ 001 — 0 candidates | bw_used=0.100


""



⚪ 010 — 1 candidates | bw_used=0.100


...

,Ŝ,ΔS,error,status,interesting
0,"[89.59132, 0.00000, 82.76566, 0.50291]","[55.14861, -87.37508, 82.76566, -0.00044]",2.9345%,✅,✅
1,"[76.83192, 0.00000, 88.76920, 0.50930]","[42.38921, -87.37508, 88.76920, 0.00596]",3.5174%,✅,✅



⚪ 110 — 6 candidates | bw_used=0.300


,Ŝ,ΔS,error,status,interesting
0,"[78.23338, 79.80807, 0.00000, 0.50310]","[43.79067, -7.56700, 0.00000, -0.00025]",1.8551%,✅,✅
1,"[81.19319, 73.98308, 0.00000, 0.50211]","[46.75048, -13.39200, 0.00000, -0.00124]",2.0997%,✅,✅
2,"[40.50713, 86.00661, 0.00000, 0.50385]","[6.06442, -1.36846, 0.00000, 0.00050]",1.5497%,✅,✅
3,"[55.30844, 86.30849, 0.00000, 0.50540]","[20.86573, -1.06659, 0.00000, 0.00205]",1.6663%,✅,✅
4,"[84.35855, 44.10110, 0.00000, 0.50093]","[49.91584, -43.27397, 0.00000, -0.00241]",3.0648%,✅,✅
5,"[83.56264, 53.60579, 0.00000, 0.50092]","[49.11993, -33.76929, 0.00000, -0.00242]",2.9157%,✅,✅



⚪ 111 — 4 candidates | bw_used=0.700


,Ŝ,ΔS,error,status,interesting
0,"[67.16125, 69.36722, 35.40046, 0.50783]","[32.71854, -18.00786, 35.40046, 0.00448]",2.5878%,✅,✅
1,"[45.95580, 72.07851, 58.39887, 0.50760]","[11.51309, -15.29657, 58.39887, 0.00425]",3.2457%,✅,✅
2,"[70.65545, 45.41107, 73.26733, 0.50726]","[36.21274, -41.96401, 73.26733, 0.00391]",3.6509%,✅,✅
3,"[51.85742, 75.08183, 44.56200, 0.50624]","[17.41471, -12.29325, 44.56200, 0.00289]",2.1797%,✅,✅
